# 第二阶段：全维度特征体系构建

本 Notebook 基于清洗后的酒店预订数据构建建模候选特征，覆盖时间、预订行为、客户群体、价格、收益管理、风险识别和业务评分等维度。

注意：原始数据是订单级数据，缺少真实 `user_id`。因此本阶段涉及“用户”的特征均为客户属性组合层面的近似分析，不代表单个真实用户。

## 1. 导入工具库与路径配置

In [1]:
# 配置输入输出路径，后续只使用项目相对路径读写数据。
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "hotel_bookings_cleaned.parquet"
FEATURE_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "hotel_booking_features.parquet"
FEATURE_DICTIONARY_PATH = PROJECT_ROOT / "reports" / "feature_dictionary.csv"

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

INPUT_PATH, FEATURE_OUTPUT_PATH, FEATURE_DICTIONARY_PATH

(WindowsPath('D:/携程实习文件/hotel-booking-analysis/data/processed/hotel_bookings_cleaned.parquet'),
 WindowsPath('D:/携程实习文件/hotel-booking-analysis/data/processed/hotel_booking_features.parquet'),
 WindowsPath('D:/携程实习文件/hotel-booking-analysis/reports/feature_dictionary.csv'))

## 2. 读取清洗后数据

In [2]:
# 读取第一阶段清洗后的 Parquet 数据，并统一关键日期字段类型。
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"清洗后数据不存在：{INPUT_PATH}")

df = pd.read_parquet(INPUT_PATH)
df["arrival_date"] = pd.to_datetime(df["arrival_date"])
df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"])

print(f"清洗后数据规模：{df.shape[0]:,} 行，{df.shape[1]:,} 列")
df.head()

清洗后数据规模：110,628 行，35 列


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,city,total_stays,arrival_date
0,Resort Hotel - Chandigarh,0,342,2024,July,30,27,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,0.0,0.0,0,Transient,0.0,0,0,Check-Out,2024-07-27 22:16:40.916332324,Chandigarh,0,2024-07-27
1,Resort Hotel - Mumbai,0,737,2024,April,17,28,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,0.0,0.0,0,Transient,0.0,0,0,Check-Out,2024-04-28 21:56:21.507509066,Mumbai,0,2024-04-28
2,Resort Hotel - Delhi,0,7,2024,September,37,10,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,0.0,0.0,0,Transient,75.0,0,0,Check-Out,2024-09-10 03:46:25.734029096,Delhi,1,2024-09-10
3,Resort Hotel - Kolkata,0,13,2024,August,33,14,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,0.0,0,Transient,75.0,0,0,Check-Out,2024-08-14 18:07:10.049669568,Kolkata,1,2024-08-14
4,Resort Hotel - Lucknow,0,14,2024,September,37,14,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,0.0,0,Transient,98.0,0,1,Check-Out,2024-09-14 14:27:32.473846000,Lucknow,2,2024-09-14


## 3. 特征构建辅助函数

这里的历史统计尽量使用当前订单之前的累计信息，避免直接使用全量数据的目标变量结果。由于数据没有真实预订时间，本阶段使用 `arrival_date` 作为历史顺序的近似依据。

In [3]:
# 定义通用辅助函数，用于百分位打分和历史统计特征构造。
def percentile_score(series, higher_is_better=True, score_min=1, score_max=5):
    """按百分位给连续变量打分。"""
    percentile = series.rank(pct=True, method="average")
    if not higher_is_better:
        percentile = 1 - percentile + (1 / len(series))
    score = np.ceil(percentile * score_max).clip(score_min, score_max)
    return score.astype("int8")


def add_historical_rate(data, group_col, target_col, prefix):
    """按分组构造当前记录之前的历史订单数和历史取消率。"""
    ordered = data.sort_values(["arrival_date", "reservation_status_date"]).copy()
    group = ordered.groupby(group_col, observed=False)
    prior_count = group.cumcount()
    prior_target_sum = group[target_col].cumsum() - ordered[target_col]
    prior_rate = prior_target_sum / prior_count.replace(0, np.nan)

    data[f"{prefix}_hist_booking_count"] = prior_count.reindex(data.index).fillna(0).astype("int32")
    data[f"{prefix}_hist_cancel_rate"] = prior_rate.reindex(data.index).fillna(0).astype("float32")
    return data


def add_historical_mean(data, group_col, value_col, prefix):
    """按分组构造当前记录之前的历史均值。"""
    ordered = data.sort_values(["arrival_date", "reservation_status_date"]).copy()
    group = ordered.groupby(group_col, observed=False)
    prior_count = group.cumcount()
    prior_sum = group[value_col].cumsum() - ordered[value_col]
    prior_mean = prior_sum / prior_count.replace(0, np.nan)
    fallback = data[value_col].mean()

    data[f"{prefix}_hist_avg_{value_col}"] = (
        prior_mean.reindex(data.index).fillna(fallback).astype("float32")
    )
    return data

## 4. 时间特征

In [4]:
# 构造时间、周末、节假日代理和淡旺季等时间类特征。
features = df.copy()

features["hotel_type"] = np.where(
    features["hotel"].astype(str).str.startswith("City Hotel"),
    "City Hotel",
    np.where(features["hotel"].astype(str).str.startswith("Resort Hotel"), "Resort Hotel", "Unknown"),
)
features["arrival_month_num"] = features["arrival_date"].dt.month.astype("int8")
features["arrival_month"] = features["arrival_date"].dt.to_period("M").astype(str)
features["arrival_quarter"] = features["arrival_date"].dt.quarter.astype("int8")
features["arrival_quarter_label"] = features["arrival_date"].dt.to_period("Q").astype(str)
features["arrival_day_of_week"] = features["arrival_date"].dt.dayofweek.astype("int8")
features["is_weekend"] = features["arrival_day_of_week"].isin([5, 6]).astype("int8")

# 本项目当前口径中，将周末作为节假日/休闲出行日的近似代理。
features["is_holiday_proxy"] = features["is_weekend"]
features["holiday_proxy"] = np.where(features["is_weekend"] == 1, "周末/节假日代理", "工作日")
features["is_near_holiday_proxy"] = features["arrival_day_of_week"].isin([0, 4]).astype("int8")

monthly_count = features.groupby("arrival_month_num").size()
peak_threshold = monthly_count.quantile(0.75)
low_threshold = monthly_count.quantile(0.25)
peak_months = monthly_count[monthly_count >= peak_threshold].index
low_months = monthly_count[monthly_count <= low_threshold].index

features["season_type"] = np.select(
    [
        features["arrival_month_num"].isin(peak_months),
        features["arrival_month_num"].isin(low_months),
    ],
    ["旺季", "淡季"],
    default="平季",
)
features["is_peak_season"] = (features["season_type"] == "旺季").astype("int8")
features["is_low_season"] = (features["season_type"] == "淡季").astype("int8")

features[
    [
        "arrival_month_num",
        "arrival_month",
        "arrival_quarter",
        "arrival_quarter_label",
        "arrival_day_of_week",
        "is_weekend",
        "is_holiday_proxy",
        "holiday_proxy",
        "is_near_holiday_proxy",
        "season_type",
        "is_peak_season",
        "is_low_season",
    ]
].head()

,arrival_month_num,arrival_month,arrival_quarter,arrival_quarter_label,arrival_day_of_week,is_weekend,is_holiday_proxy,holiday_proxy,is_near_holiday_proxy,season_type,is_peak_season,is_low_season
0,7,2024-07,3,2024Q3,5,1,1,周末/节假日代理,0,旺季,1,0
1,4,2024-04,2,2024Q2,6,1,1,周末/节假日代理,0,平季,0,0
2,9,2024-09,3,2024Q3,1,0,0,工作日,0,平季,0,0
3,8,2024-08,3,2024Q3,2,0,0,工作日,0,平季,0,0
4,9,2024-09,3,2024Q3,5,1,1,周末/节假日代理,0,平季,0,0


## 5. 预订行为特征

In [5]:
# 构造预订行为特征，包括总入住晚数、总客人数、房型匹配和需求分组。
features["total_stays"] = (
    features["stays_in_weekend_nights"] + features["stays_in_week_nights"]
).astype("int16")
features["total_guests"] = (features["adults"] + features["children"] + features["babies"]).astype("float32")
features["has_children_or_babies"] = ((features["children"] > 0) | (features["babies"] > 0)).astype("int8")
features["room_type_matched"] = (features["reserved_room_type"].astype(str) == features["assigned_room_type"].astype(str)).astype("int8")
features["has_booking_changes"] = (features["booking_changes"] > 0).astype("int8")
features["has_special_requests"] = (features["total_of_special_requests"] > 0).astype("int8")
features["has_car_parking_request"] = (features["required_car_parking_spaces"] > 0).astype("int8")
features["lead_time_group"] = pd.cut(
    features["lead_time"],
    bins=[-1, 7, 30, 90, 180, np.inf],
    labels=["0-7", "8-30", "31-90", "91-180", "181+"],
)
features["special_request_group"] = features["total_of_special_requests"].clip(upper=5).astype(int).astype(str)
features.loc[features["total_of_special_requests"] >= 5, "special_request_group"] = "5+"

features[
    [
        "total_stays",
        "total_guests",
        "has_children_or_babies",
        "room_type_matched",
        "has_booking_changes",
        "has_special_requests",
        "has_car_parking_request",
        "lead_time_group",
        "special_request_group",
    ]
].head()

,total_stays,total_guests,has_children_or_babies,room_type_matched,has_booking_changes,has_special_requests,has_car_parking_request,lead_time_group,special_request_group
0,0,2.0,0,1,1,0,0,181+,0
1,0,2.0,0,1,1,0,0,181+,0
2,1,1.0,0,0,0,0,0,0-7,0
3,1,1.0,0,1,0,0,0,8-30,0
4,2,2.0,0,1,0,1,0,8-30,1


## 6. 客户群体与 RFM 特征

由于没有真实 `user_id`，这里用 `customer_type`、`market_segment`、`distribution_channel`、`is_repeated_guest` 组合成客户群体。

In [6]:
# 原始数据没有真实 user_id，因此按客户属性组合构造客户群体键和 RFM 近似特征。
customer_group_cols = ["customer_type", "market_segment", "distribution_channel", "is_repeated_guest"]
features["customer_group_key"] = (
    features[customer_group_cols].astype(str).agg("|".join, axis=1)
)
features["estimated_revenue"] = features["adr"] * features["total_stays"].clip(lower=1)

rfm = (
    features.groupby("customer_group_key")
    .agg(
        group_booking_count=("is_canceled", "size"),
        group_avg_revenue=("estimated_revenue", "mean"),
        group_total_revenue=("estimated_revenue", "sum"),
        last_arrival_date=("arrival_date", "max"),
    )
    .reset_index()
)
max_arrival_date = features["arrival_date"].max()
rfm["group_recency_days"] = (max_arrival_date - rfm["last_arrival_date"]).dt.days
rfm["r_score"] = percentile_score(rfm["group_recency_days"], higher_is_better=False)
rfm["f_score"] = percentile_score(rfm["group_booking_count"], higher_is_better=True)
rfm["m_score"] = percentile_score(rfm["group_avg_revenue"], higher_is_better=True)
rfm["rfm_value_score"] = rfm["r_score"] + rfm["f_score"] + rfm["m_score"]

q25, q50, q75 = rfm["rfm_value_score"].quantile([0.25, 0.5, 0.75])
rfm["rfm_segment"] = np.select(
    [
        rfm["rfm_value_score"] >= q75,
        rfm["rfm_value_score"] >= q50,
        rfm["rfm_value_score"] >= q25,
    ],
    ["高价值", "潜力", "沉睡"],
    default="流失",
)
rfm["is_high_value_customer_group"] = (rfm["rfm_segment"] == "高价值").astype("int8")

features = features.merge(
    rfm[
        [
            "customer_group_key",
            "group_recency_days",
            "group_booking_count",
            "group_avg_revenue",
            "rfm_value_score",
            "rfm_segment",
            "is_high_value_customer_group",
        ]
    ],
    on="customer_group_key",
    how="left",
)

rfm["rfm_segment"].value_counts()

rfm_segment
高价值    38
沉睡     37
潜力     25
流失     21
Name: count, dtype: int64

## 7. 历史行为与风险特征

`hist_cancel_rate` 类特征使用当前订单之前的累计取消率。这类特征仍然要在建模阶段谨慎使用，训练/验证/测试划分后需要确保不会跨集合泄露。

In [7]:
# 按时间顺序计算历史取消率，避免直接使用全量目标均值造成明显泄露。
features = add_historical_rate(features, "customer_group_key", "is_canceled", "customer_group")
features = add_historical_rate(features, "country", "is_canceled", "country")
features = add_historical_rate(features, "agent", "is_canceled", "agent")
features = add_historical_rate(features, "hotel", "is_canceled", "hotel")

country_risk_threshold = features.loc[
    features["country_hist_booking_count"] >= 50, "country_hist_cancel_rate"
].quantile(0.75)
agent_risk_threshold = features.loc[
    features["agent_hist_booking_count"] >= 30, "agent_hist_cancel_rate"
].quantile(0.75)

features["is_high_risk_country"] = (
    (features["country_hist_booking_count"] >= 50)
    & (features["country_hist_cancel_rate"] >= country_risk_threshold)
).astype("int8")
features["is_high_risk_agent"] = (
    (features["agent_hist_booking_count"] >= 30)
    & (features["agent_hist_cancel_rate"] >= agent_risk_threshold)
).astype("int8")

risk_cols = [
    "customer_group_hist_booking_count",
    "customer_group_hist_cancel_rate",
    "country_hist_booking_count",
    "country_hist_cancel_rate",
    "agent_hist_booking_count",
    "agent_hist_cancel_rate",
    "hotel_hist_cancel_rate",
    "is_high_risk_country",
    "is_high_risk_agent",
]
features[risk_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
customer_group_hist_booking_count,110628.0,11602.849233,13034.574104,0.0,1736.000000,6040.000000,18233.250000,45890.0
customer_group_hist_cancel_rate,110628.0,0.371344,0.209669,0.0,0.252529,0.380306,0.383699,1.0
country_hist_booking_count,110628.0,11173.331345,13257.619989,0.0,1189.000000,5145.000000,18243.250000,45900.0
country_hist_cancel_rate,110628.0,0.370855,0.178670,0.0,0.209869,0.300676,0.566926,1.0
agent_hist_booking_count,110628.0,6380.537405,7885.600933,0.0,383.000000,2752.000000,9922.250000,30515.0
agent_hist_cancel_rate,110628.0,0.371140,0.194807,0.0,0.249033,0.372673,0.414496,1.0
hotel_hist_cancel_rate,110628.0,0.371271,0.071575,0.0,0.279891,0.410580,0.420103,1.0
is_high_risk_country,110628.0,0.240373,0.427312,0.0,0.000000,0.000000,0.000000,1.0
is_high_risk_agent,110628.0,0.236522,0.424948,0.0,0.000000,0.000000,0.000000,1.0


## 8. 价格特征

In [8]:
# 基于历史 ADR 构造价格相对水平、价格波动和价格敏感度特征。
features = add_historical_mean(features, "hotel", "adr", "hotel")
features = add_historical_mean(features, "city", "adr", "city")
features = add_historical_mean(features, "customer_group_key", "adr", "customer_group")

features["price_to_hotel_hist_avg"] = (
    features["adr"] / features["hotel_hist_avg_adr"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(1).astype("float32")
features["price_to_city_hist_avg"] = (
    features["adr"] / features["city_hist_avg_adr"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(1).astype("float32")
features["price_to_customer_group_hist_avg"] = (
    features["adr"] / features["customer_group_hist_avg_adr"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(1).astype("float32")

hotel_month_price = (
    features.groupby(["hotel", "arrival_month_num"], observed=False)["adr"]
    .agg(["mean", "std"])
    .reset_index()
    .rename(columns={"mean": "hotel_month_avg_adr", "std": "hotel_month_std_adr"})
)
hotel_month_price["price_volatility_index"] = (
    hotel_month_price["hotel_month_std_adr"]
    / hotel_month_price["hotel_month_avg_adr"].replace(0, np.nan)
).fillna(0)

features = features.merge(
    hotel_month_price[["hotel", "arrival_month_num", "price_volatility_index"]],
    on=["hotel", "arrival_month_num"],
    how="left",
)
features["price_volatility_index"] = features["price_volatility_index"].fillna(0).astype("float32")
features["price_sensitivity_score"] = percentile_score(
    features["price_to_customer_group_hist_avg"].astype("float64"),
    higher_is_better=True,
)

features[
    [
        "estimated_revenue",
        "hotel_hist_avg_adr",
        "price_to_hotel_hist_avg",
        "price_to_customer_group_hist_avg",
        "price_volatility_index",
        "price_sensitivity_score",
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
estimated_revenue,110628.0,302.320038,225.857346,0.000000,136.000000,252.000000,397.799988,1477.209961
hotel_hist_avg_adr,110628.0,96.995834,9.156330,37.800003,84.348219,102.179367,103.240469,202.000000
price_to_hotel_hist_avg,110628.0,1.000633,0.425380,0.000000,0.719349,0.935492,1.223367,3.532000
price_to_customer_group_hist_avg,110628.0,1.001660,0.629000,0.000000,0.754500,0.971259,1.225691,91.949699
price_volatility_index,110628.0,0.411313,0.104585,0.305062,0.337709,0.351131,0.544133,0.643544
price_sensitivity_score,110628.0,3.000018,1.414220,1.000000,2.000000,3.000000,4.000000,5.000000


## 9. 收益管理特征

原始数据没有酒店真实房间库存，因此这里构造的是需求/入住压力代理变量，不是真实入住率。

In [9]:
# 基于同酒店同入住日的订单量构造需求紧张程度和入住率代理变量。
hotel_day_demand = (
    features.groupby(["hotel", "arrival_date"], observed=False)
    .agg(hotel_daily_booking_count=("is_canceled", "size"))
    .reset_index()
)
hotel_day_demand["hotel_max_daily_booking_count"] = hotel_day_demand.groupby("hotel", observed=False)[
    "hotel_daily_booking_count"
].transform("max")
hotel_day_demand["estimated_occupancy_proxy"] = (
    hotel_day_demand["hotel_daily_booking_count"]
    / hotel_day_demand["hotel_max_daily_booking_count"].replace(0, np.nan)
).fillna(0)

features = features.merge(
    hotel_day_demand[
        ["hotel", "arrival_date", "hotel_daily_booking_count", "estimated_occupancy_proxy"]
    ],
    on=["hotel", "arrival_date"],
    how="left",
)
features["demand_pressure_score"] = percentile_score(
    features["hotel_daily_booking_count"].astype("float64"),
    higher_is_better=True,
)

features[
    [
        "hotel_daily_booking_count",
        "estimated_occupancy_proxy",
        "demand_pressure_score",
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
hotel_daily_booking_count,110628.0,12.633926,4.973144,1.000000,9.000000,13.000000,16.000000,30.0
estimated_occupancy_proxy,110628.0,0.544035,0.161474,0.037037,0.428571,0.538462,0.653846,1.0
demand_pressure_score,110628.0,2.954433,1.361427,1.000000,2.000000,3.000000,4.000000,5.0


## 10. 业务导向评分特征

In [10]:
# 综合历史取消率、提前预订天数、押金类型和特殊需求构造取消风险评分。
lead_time_score = features["lead_time"].rank(pct=True).astype("float32")
deposit_risk_score = features["deposit_type"].astype(str).map(
    {"No Deposit": 0.3, "Refundable": 0.5, "Non Refund": 0.8}
).fillna(0.3).astype("float32")
special_request_protection = (
    1 - features["total_of_special_requests"].clip(upper=5) / 5
).astype("float32")

features["cancellation_risk_score"] = (
    0.30 * features["customer_group_hist_cancel_rate"]
    + 0.20 * features["country_hist_cancel_rate"]
    + 0.15 * features["agent_hist_cancel_rate"]
    + 0.15 * lead_time_score
    + 0.10 * deposit_risk_score
    + 0.10 * special_request_protection
).clip(0, 1).astype("float32")

score_cols = [
    "rfm_value_score",
    "price_sensitivity_score",
    "cancellation_risk_score",
    "demand_pressure_score",
]
features[score_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
rfm_value_score,110628.0,14.654662,0.936776,3.000000,15.000000,15.000000,15.00000,15.000000
price_sensitivity_score,110628.0,3.000018,1.414220,1.000000,2.000000,3.000000,4.00000,5.000000
cancellation_risk_score,110628.0,0.441693,0.134212,0.111188,0.357687,0.412695,0.47417,0.885572
demand_pressure_score,110628.0,2.954433,1.361427,1.000000,2.000000,3.000000,4.00000,5.000000


## 11. 特征完整性检查

In [11]:
# 检查新增特征的缺失率和字段类型，确认特征工程输出质量。
new_feature_cols = [
    col for col in features.columns if col not in df.columns
]
feature_check = pd.DataFrame(
    {
        "feature_name": new_feature_cols,
        "missing_count": [features[col].isna().sum() for col in new_feature_cols],
        "missing_rate": [features[col].isna().mean() for col in new_feature_cols],
        "dtype": [str(features[col].dtype) for col in new_feature_cols],
    }
).sort_values("missing_rate", ascending=False)

print(f"原始清洗数据列数：{df.shape[1]}")
print(f"特征构建后列数：{features.shape[1]}")
print(f"新增特征数量：{len(new_feature_cols)}")
feature_check.head(20)

原始清洗数据列数：35
特征构建后列数：86
新增特征数量：51


,feature_name,missing_count,missing_rate,dtype
0,hotel_type,0,0.0,str
1,arrival_month_num,0,0.0,int8
2,arrival_month,0,0.0,str
3,arrival_quarter,0,0.0,int8
4,arrival_quarter_label,0,0.0,str
5,arrival_day_of_week,0,0.0,int8
6,is_weekend,0,0.0,int8
7,is_holiday_proxy,0,0.0,int8
8,holiday_proxy,0,0.0,str
9,is_near_holiday_proxy,0,0.0,int8


## 12. 保存特征数据与特征字典

In [12]:
# 手工维护特征字典，说明字段来源、业务含义和潜在限制。
feature_dictionary_rows = [
    ("arrival_month_num", "时间特征", "arrival_date", "入住月份数字。", "无"),
    ("arrival_month", "时间特征", "arrival_date", "入住年月，格式为 YYYY-MM。", "无"),
    ("arrival_quarter_label", "时间特征", "arrival_date", "入住季度标签，格式为 YYYYQn。", "无"),
    ("arrival_quarter", "时间特征", "arrival_date", "入住季度。", "无"),
    ("arrival_day_of_week", "时间特征", "arrival_date", "入住日期星期几，0 表示周一。", "无"),
    ("is_weekend", "时间特征", "arrival_date", "是否周末。", "无"),
    ("is_holiday_proxy", "时间特征", "is_weekend", "按项目口径将周末作为节假日/休闲出行日代理。", "不是法定节假日。"),
    ("holiday_proxy", "时间特征", "is_weekend", "周末节假日代理标签，用于 EDA 分组展示。", "不是法定节假日。"),
    ("is_near_holiday_proxy", "时间特征", "arrival_day_of_week", "周五和周一作为靠近周末节假日的代理。", "不是法定节假日前后。"),
    ("season_type", "季节性特征", "arrival_month_num", "基于月度预订量分位数划分淡季、平季、旺季。", "仅为当前样本探索口径。"),
    ("is_peak_season", "季节性特征", "season_type", "是否旺季。", "仅为当前样本探索口径。"),
    ("is_low_season", "季节性特征", "season_type", "是否淡季。", "仅为当前样本探索口径。"),
    ("total_guests", "预订特征", "adults/children/babies", "订单总客人数。", "无"),
    ("has_children_or_babies", "预订特征", "children/babies", "是否带儿童或婴儿。", "无"),
    ("room_type_matched", "预订特征", "reserved_room_type/assigned_room_type", "预订房型与实际分配房型是否一致。", "建模时需确认该字段在预测时是否已知。"),
    ("has_booking_changes", "预订特征", "booking_changes", "是否发生过预订变更。", "建模时需确认预测时点。"),
    ("has_special_requests", "预订特征", "total_of_special_requests", "是否有特殊需求。", "无"),
    ("has_car_parking_request", "预订特征", "required_car_parking_spaces", "是否需要停车位。", "无"),
    ("lead_time_group", "预订特征", "lead_time", "提前预订天数分组。", "用于 EDA 分组和候选建模特征。"),
    ("special_request_group", "预订特征", "total_of_special_requests", "特殊需求数量分组，5 个及以上合并为 5+。", "用于 EDA 分组和候选建模特征。"),
    ("customer_group_key", "客户群体特征", "customer_type/market_segment/distribution_channel/is_repeated_guest", "客户属性组合键。", "不是真实 user_id。"),
    ("rfm_value_score", "业务评分特征", "客户群体聚合", "客户群体层面的 RFM 价值分。", "不是真实用户级 RFM。"),
    ("rfm_segment", "业务评分特征", "rfm_value_score", "客户群体 RFM 分层。", "不是真实用户级分层。"),
    ("is_high_value_customer_group", "客户群体特征", "rfm_segment", "是否高价值客户群体。", "不是单个真实用户。"),
    ("customer_group_hist_cancel_rate", "风险特征", "历史累计", "客户群体历史取消率。", "目标衍生特征，建模需严格防泄露。"),
    ("country_hist_cancel_rate", "风险特征", "历史累计", "国家/地区历史取消率。", "目标衍生特征，建模需严格防泄露。"),
    ("agent_hist_cancel_rate", "风险特征", "历史累计", "代理商历史取消率。", "目标衍生特征，建模需严格防泄露。"),
    ("is_high_risk_country", "风险特征", "country_hist_cancel_rate", "是否高风险国家/地区。", "目标衍生特征，建模需严格防泄露。"),
    ("is_high_risk_agent", "风险特征", "agent_hist_cancel_rate", "是否高风险代理商。", "目标衍生特征，建模需严格防泄露。"),
    ("estimated_revenue", "价格特征", "adr/total_stays", "估算消费金额。", "不是实际支付金额。"),
    ("price_to_hotel_hist_avg", "价格特征", "adr/酒店历史 ADR", "当前 ADR 与酒店历史平均 ADR 的比值。", "无"),
    ("price_to_city_hist_avg", "价格特征", "adr/城市历史 ADR", "当前 ADR 与城市历史平均 ADR 的比值。", "无"),
    ("price_to_customer_group_hist_avg", "价格特征", "adr/客户群体历史 ADR", "当前 ADR 与客户群体历史平均 ADR 的比值。", "不是真实用户历史价格。"),
    ("price_volatility_index", "价格特征", "hotel/month adr", "酒店月度价格波动系数。", "无"),
    ("price_sensitivity_score", "业务评分特征", "price_to_customer_group_hist_avg", "价格敏感度近似得分。", "基于客户群体，不是用户级。"),
    ("hotel_daily_booking_count", "收益管理特征", "hotel/arrival_date", "酒店到店日订单量。", "不是入住率。"),
    ("estimated_occupancy_proxy", "收益管理特征", "hotel_daily_booking_count", "以酒店日订单量相对峰值近似需求压力。", "不是实际入住率。"),
    ("demand_pressure_score", "收益管理特征", "hotel_daily_booking_count", "需求紧张程度分位得分。", "不是实际库存压力。"),
    ("cancellation_risk_score", "业务评分特征", "多字段组合", "结合历史取消、提前预订和特殊需求的探索性取消风险评分。", "目标衍生评分，建模需严格防泄露。"),
]

feature_dictionary = pd.DataFrame(
    feature_dictionary_rows,
    columns=["feature_name", "feature_category", "source", "description", "leakage_or_limit_note"],
)

FEATURE_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
FEATURE_DICTIONARY_PATH.parent.mkdir(parents=True, exist_ok=True)

features.to_parquet(FEATURE_OUTPUT_PATH, index=False)
feature_dictionary.to_csv(FEATURE_DICTIONARY_PATH, index=False, encoding="utf-8-sig")

print(f"特征数据已保存：{FEATURE_OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"特征字典已保存：{FEATURE_DICTIONARY_PATH.relative_to(PROJECT_ROOT)}")
print(f"特征数据规模：{features.shape[0]:,} 行，{features.shape[1]:,} 列")
feature_dictionary.head(10)

特征数据已保存：data\processed\hotel_booking_features.parquet
特征字典已保存：reports\feature_dictionary.csv
特征数据规模：110,628 行，86 列


,feature_name,feature_category,source,description,leakage_or_limit_note
0,arrival_month_num,时间特征,arrival_date,入住月份数字。,无
1,arrival_month,时间特征,arrival_date,入住年月，格式为 YYYY-MM。,无
2,arrival_quarter_label,时间特征,arrival_date,入住季度标签，格式为 YYYYQn。,无
3,arrival_quarter,时间特征,arrival_date,入住季度。,无
4,arrival_day_of_week,时间特征,arrival_date,入住日期星期几，0 表示周一。,无
5,is_weekend,时间特征,arrival_date,是否周末。,无
6,is_holiday_proxy,时间特征,is_weekend,按项目口径将周末作为节假日/休闲出行日代理。,不是法定节假日。
7,holiday_proxy,时间特征,is_weekend,周末节假日代理标签，用于 EDA 分组展示。,不是法定节假日。
8,is_near_holiday_proxy,时间特征,arrival_day_of_week,周五和周一作为靠近周末节假日的代理。,不是法定节假日前后。
9,season_type,季节性特征,arrival_month_num,基于月度预订量分位数划分淡季、平季、旺季。,仅为当前样本探索口径。


## 13. 阶段性说明

In [13]:
# 输出本阶段总结，提醒后续建模时重点关注数据泄露和代理变量限制。
summary = [
    "已构建时间、预订行为、客户群体、价格、收益管理、风险识别和业务评分等多类候选特征。",
    "原始数据缺少真实 user_id，因此用户特征和 RFM 均为客户群体层面的近似构造。",
    "历史取消率、高风险国家/代理商、取消风险评分属于目标衍生特征，后续建模时必须在训练集内部重新计算或使用严格的时间顺序计算，避免数据泄露。",
    "入住率预估和需求紧张程度是基于订单量的代理变量，不代表真实房间库存或真实入住率。",
    "节假日相关特征按当前项目口径使用周末代理，不代表法定节假日。",
]

for item in summary:
    print("- " + item)

- 已构建时间、预订行为、客户群体、价格、收益管理、风险识别和业务评分等多类候选特征。
- 原始数据缺少真实 user_id，因此用户特征和 RFM 均为客户群体层面的近似构造。
- 历史取消率、高风险国家/代理商、取消风险评分属于目标衍生特征，后续建模时必须在训练集内部重新计算或使用严格的时间顺序计算，避免数据泄露。
- 入住率预估和需求紧张程度是基于订单量的代理变量，不代表真实房间库存或真实入住率。
- 节假日相关特征按当前项目口径使用周末代理，不代表法定节假日。
